# Lab 1.5 &mdash; Challenge &mdash; The Decision Rubric, Made Executable

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 1 &middot; Module 1 &mdash; Agents vs. Multi-Agent Systems**

### What you'll do
- Encode the four-question rubric as code that returns a defensible verdict
- Run it over six real-shaped briefs, including two designed to mislead
- Gate the verdict on evidence: a budget named BEFORE you measured
- Produce a recommendation you could defend in a design review

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, so your score never depends on a
> live endpoint. Cells marked **Run it for real** do call the sandbox model; if it is not
> reachable they print how to fix it instead of crashing.

> **The comprehensive lab for Module 1.** It uses the rubric from the slides, the
> scorecard from Lab 1.4, and the honesty that the two together are supposed to enforce.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-1-05")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- graded cells still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 1 labs: payment exceptions on a small ledger.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

## Concept

The rubric picks a **candidate**, not a winner. A candidate becomes a decision only when the
scorecard says it earned its cost &mdash; against a threshold you wrote down **before** you measured,
because a number chosen afterwards can always be argued into looking acceptable.

This lab makes both halves executable.

## Section 1 &mdash; The rubric, as code

Four questions, in order. The order matters: most real briefs terminate at the first or second.

In [ ]:
VERDICTS = ("workflow", "single_agent", "supervisor_worker", "peer_to_peer")

def rubric(brief: dict) -> str:
    """Return one of VERDICTS for a brief.

    brief keys:
      steps_vary       -- do the steps change with the input?
      fits_one_agent   -- can one agent hold every tool it needs? (roughly a dozen)
      different_tools  -- do the sub-tasks need genuinely different tools/permissions?
      parallel         -- can the sub-tasks genuinely run at the same time?
      route_varies     -- does the path through the work change per case?
    """
    if BLANK:                          # TODO: question 1 -- are the steps the same every time?
        return "workflow"
    if brief["fits_one_agent"]:
        return "single_agent"
    if BLANK:                          # TODO: question 3 -- neither specialisation nor parallelism?
        return "single_agent"        #        then splitting buys nothing measurable
    return BLANK                       # TODO: question 4 -- varying route or fixed?

In [ ]:
# --- Self-check: Section 1
BRIEFS = [
    {"name": "nightly ledger reconciliation", "steps_vary": False, "fits_one_agent": True,
     "different_tools": False, "parallel": False, "route_varies": False},
    {"name": "policy Q&A with citations", "steps_vary": True, "fits_one_agent": True,
     "different_tools": False, "parallel": False, "route_varies": False},
    # designed to mislead: sounds big, but splitting buys nothing
    {"name": "three-stage summary pipeline", "steps_vary": True, "fits_one_agent": False,
     "different_tools": False, "parallel": False, "route_varies": False},
    {"name": "payment exception investigation", "steps_vary": True, "fits_one_agent": False,
     "different_tools": True, "parallel": True, "route_varies": False},
    {"name": "open-ended client complaint triage", "steps_vary": True, "fits_one_agent": False,
     "different_tools": True, "parallel": False, "route_varies": True},
    # designed to mislead: varying route, but it still fits in one agent
    {"name": "ad-hoc data question over one warehouse", "steps_vary": True, "fits_one_agent": True,
     "different_tools": False, "parallel": False, "route_varies": True},
]

for b in BRIEFS:
    try:
        print(f"{b['name']:38} -> {rubric(b)}")
    except NameError:
        print("(fill in rubric() above)")
        break

check("fixed steps give a workflow", lambda: rubric(BRIEFS[0]) == "workflow")
check("one skill, one source gives a single agent", lambda: rubric(BRIEFS[1]) == "single_agent")
check("splitting without specialisation or parallelism falls back to one agent",
      lambda: rubric(BRIEFS[2]) == "single_agent",
      "question 3 is the one that catches this brief")
check("different tools + parallel + fixed route gives supervisor/worker",
      lambda: rubric(BRIEFS[3]) == "supervisor_worker")
check("a varying route gives peer to peer", lambda: rubric(BRIEFS[4]) == "peer_to_peer")
check("a varying route that still fits one agent stays a single agent",
      lambda: rubric(BRIEFS[5]) == "single_agent",
      "question 2 comes before question 4 for a reason")

## Section 2 &mdash; The budget, written down first

Name the thresholds now, while you have no result to defend. A multi-agent design must beat the
single agent by at least `min_gain` **and** stay inside the cost and latency ceilings.

In [ ]:
BUDGET = {
    "min_gain": 0.10,        # pass-rate points the new design must add, as a fraction
    "max_token_ratio": 3.0,  # at most 3x the single agent's tokens
    "max_latency_ratio": 2.0
}

def clears_budget(single: dict, multi: dict, budget: dict = BUDGET) -> tuple[bool, str]:
    """Return (cleared, reason). Every clause must hold for the multi-agent design to win."""
    gain = multi["pass_rate"] - single["pass_rate"]
    token_ratio = multi["tokens"] / max(single["tokens"], 1)
    latency_ratio = multi["seconds"] / max(single["seconds"], 1e-9)

    if gain < budget["min_gain"]:
        return False, f"gain {gain:+.2f} is below the {budget['min_gain']:.2f} threshold"
    if BLANK:                          # TODO: is it over the token ceiling?
        return False, f"tokens {token_ratio:.1f}x exceed {budget['max_token_ratio']}x"
    if latency_ratio > budget["max_latency_ratio"]:
        return False, f"latency {latency_ratio:.1f}x exceeds {budget['max_latency_ratio']}x"
    return True, f"gain {gain:+.2f} within {token_ratio:.1f}x tokens"

In [ ]:
# --- Self-check: Section 2
_base = {"pass_rate": 0.80, "tokens": 1000, "seconds": 1.0}
_worse_cost = {"pass_rate": 0.95, "tokens": 9000, "seconds": 1.5}   # big gain, absurd cost
_no_gain    = {"pass_rate": 0.82, "tokens": 1500, "seconds": 1.2}   # cheap, but no real gain
_good       = {"pass_rate": 0.95, "tokens": 2500, "seconds": 1.5}

check("a design that gains little is rejected", lambda: clears_budget(_base, _no_gain)[0] is False)
check("a design that costs too many tokens is rejected",
      lambda: clears_budget(_base, _worse_cost)[0] is False,
      "a 15-point gain does not license a 9x bill -- that is what the ceiling is for")
check("a design that clears every clause wins", lambda: clears_budget(_base, _good)[0] is True)
check("the rejection always carries a reason",
      lambda: len(clears_budget(_base, _no_gain)[1]) > 10)

## Section 3 &mdash; Candidate, then evidence

Put the two halves together. The rubric proposes; the scorecard disposes. Note the asymmetry:
when there is no evidence yet, the honest answer is the **cheaper** design, not the interesting
one.

In [ ]:
def recommend(brief: dict, single: dict | None = None, multi: dict | None = None) -> dict:
    """Return {"candidate", "decision", "why"} for one brief.

    With no measurements, the decision falls back to the cheaper design and says so.
    """
    candidate = rubric(brief)

    if candidate in ("workflow", "single_agent"):
        return {"candidate": candidate, "decision": candidate,
                "why": "the rubric terminates before multi-agent is on the table"}

    if single is None or multi is None:
        return {"candidate": candidate, "decision": BLANK,     # TODO: no evidence yet -- what ships?
                "why": "no scorecard yet; the cheaper design holds until the numbers exist"}

    cleared, reason = clears_budget(single, multi)
    return {"candidate": candidate,
            "decision": candidate if cleared else "single_agent",
            "why": reason}

In [ ]:
# --- Self-check: Section 3
_inv = BRIEFS[3]                       # payment exception investigation
_measured_bad = ({"pass_rate": 0.83, "tokens": 1000, "seconds": 1.0},
                 {"pass_rate": 0.83, "tokens": 8000, "seconds": 3.0})
_measured_good = ({"pass_rate": 0.70, "tokens": 1000, "seconds": 1.0},
                  {"pass_rate": 0.92, "tokens": 2400, "seconds": 1.6})

check("with no evidence, the cheaper design ships",
      lambda: recommend(_inv)["decision"] == "single_agent",
      "an unmeasured multi-agent design is a hypothesis, not a decision")
check("with no evidence, the candidate is still reported",
      lambda: recommend(_inv)["candidate"] == "supervisor_worker",
      "record what the rubric proposed even when you do not ship it")
check("evidence that fails the budget sends you back to one agent",
      lambda: recommend(_inv, *_measured_bad)["decision"] == "single_agent")
check("evidence that clears the budget promotes the candidate",
      lambda: recommend(_inv, *_measured_good)["decision"] == "supervisor_worker")
check("a workflow brief never reaches the scorecard",
      lambda: recommend(BRIEFS[0], *_measured_good)["decision"] == "workflow")

## Section 4 &mdash; The design-review table

One table, six briefs. This is the artefact you take back to work.

In [ ]:
def review_table(briefs, single=None, multi=None) -> str:
    """A fixed-width table of candidate vs decision for every brief."""
    rows = [f"{'brief':38} {'candidate':20} {'decision':20} why"]
    rows.append("-" * 110)
    for b in briefs:
        r = recommend(b, single, multi)
        rows.append(f"{b['name']:38} {r['candidate']:20} {r['decision']:20} {r['why']}")
    return "\n".join(rows)

try:
    print(review_table(BRIEFS))
    print()
    print(review_table([BRIEFS[3]], *_measured_good))
except NameError:
    print("(finish the sections above, then re-run this cell)")

In [ ]:
# --- Self-check: Section 4
check("the table has a row per brief plus a header and rule",
      lambda: len(review_table(BRIEFS).splitlines()) == len(BRIEFS) + 2)
check("every brief resolves to a real verdict",
      lambda: all(recommend(b)["decision"] in VERDICTS for b in BRIEFS))
check("only briefs whose steps vary escape 'workflow'",
      lambda: all((recommend(b)["decision"] == "workflow") == (not b["steps_vary"]) for b in BRIEFS))

## Run it for real

Have the model write the paragraph you would put in front of a design review. Notice what you are
asking it to do: not to *decide*, but to explain a decision your code already made and can defend.

In [ ]:
if llm_ready():
    try:
        verdict = recommend(BRIEFS[3], *_measured_good)
        summary = ask(
            "Write one short paragraph for an engineering design review. State the chosen "
            "architecture, the evidence that justified it, and the condition under which the team "
            "should revisit the decision. Be plain and specific; do not add claims beyond the "
            "facts given.\n\n"
            f"BRIEF: {BRIEFS[3]['name']}\n"
            f"CANDIDATE FROM RUBRIC: {verdict['candidate']}\n"
            f"DECISION: {verdict['decision']}\n"
            f"EVIDENCE: {verdict['why']}\n"
            f"BUDGET: {BUDGET}"
        )
        print(summary)
    except NameError:
        print("(finish the sections above, then re-run this cell)")

### Read it

If the paragraph reads as a justification you would actually sign, the rubric did its job. If it
reads as advocacy for the interesting architecture, look again at which clause let it through.

**What you take from Module 1:** a rubric that terminates early, a budget written before the
measurement, and a scorecard that can overrule your own design preference. Modules 2 and 3 make
the agent better. This lab is what stops you building one you did not need.

In [ ]:
score()

## Your turn

1. `rubric()` takes booleans, which assumes someone already made the hard calls. Replace
   `fits_one_agent` with a function of the tool count and argue for the threshold you pick.
2. Add a fifth question &mdash; **can this fail unattended?** &mdash; that can force a supervisor even when
   the rubric would otherwise say single agent. Where in the order does it belong, and why?
3. Take a real brief from your own team, fill in the five booleans honestly, and run it. If the
   verdict surprises you, which boolean were you tempted to fill in dishonestly?